# Lighter AAPL Standardized Schema Tour with `polaris_data`

This notebook complements the Hyperliquid trade walkthrough by focusing on other methods from the
`polaris_data` standardized schema surface.

It uses the public `lighter` AAPL perpetual market to demonstrate:

- `catalog(...)` to discover the numeric market id and coverage window
- `events(...)` for the normalized event stream
- `l2_snapshots(...)` for full book snapshots
- `bbo(...)` for best-bid-offer tracking
- `volume(...)`, `vwap(...)`, and `volatility(...)` for bucketed trade-derived series
- `funding_rates(...)` and `mark_prices(...)` with explicit empty-result handling when the current public window does not expose them


## Setup

This repo already pins `polaris-data`, `pandas`, and `matplotlib` in `pyproject.toml`.

Run `make install` once from the repo root, then start JupyterLab with `make notebook`.
The notebook discovers the live catalog entry first, then picks a short recent window so the public
`lighter` dataset stays fast to rerun.


In [ ]:
from polaris_data import PolarisClient
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", 20)


In [ ]:
source = "lighter"
base_asset = "AAPL"
bucket_interval = "1m"
candidate_windows = [5, 3, 1]


## Discover the Lighter AAPL market

Polaris currently exposes `lighter` markets with numeric ids, so the first step is to resolve the
AAPL instrument from catalog metadata and inspect its available coverage.


In [ ]:
with PolarisClient() as client:
    catalog = client.catalog(source=source)

markets = pd.DataFrame(catalog["markets"])
asset_matches = markets[
    markets["instrument"].apply(lambda instrument: instrument.get("base") == base_asset)
].copy()

if asset_matches.empty:
    raise ValueError(f"Could not find a {base_asset} market in {source!r}")

market_info = asset_matches.iloc[0].to_dict()
market = market_info["market"]
available_start = pd.Timestamp(market_info["start"])
available_end = pd.Timestamp(market_info["end"])

print(f"Resolved {source}:{market} for base asset {base_asset}")
print(f"Catalog coverage: {available_start} -> {available_end}")

pd.Series(market_info)


## Pick a short recent window

Public `lighter` coverage can be sparse, so the notebook tries a few short lookback windows and keeps
the first one that returns both order-book and bucketed trade-derived data.


In [ ]:
def choose_window(client, source, market, end, candidates):
    for minutes in candidates:
        start = end - pd.Timedelta(minutes=minutes)
        events = client.events(
            source=source,
            market=market,
            from_=start,
            to=end,
            allow_gaps=True,
        )
        bbo = client.bbo(
            source=source,
            market=market,
            from_=start,
            to=end,
            allow_gaps=True,
        )
        volume = client.volume(
            source=source,
            market=market,
            from_=start,
            to=end,
            interval=bucket_interval,
            allow_gaps=True,
        )
        if events and bbo and volume:
            return start, end
    raise ValueError("Could not find a recent window with populated standardized data")


with PolarisClient() as client:
    window_start, window_end = choose_window(
        client=client,
        source=source,
        market=market,
        end=available_end,
        candidates=candidate_windows,
    )

print(f"Selected analysis window: {window_start} -> {window_end}")


## Fetch the standardized schema methods

This cell pulls the normalized order-book views plus the bucketed trade-derived series for the chosen
window. `funding_rates(...)` and `mark_prices(...)` are included as well so you can see whether the
current public coverage exposes those point-series rows for `lighter`.


In [ ]:
with PolarisClient() as client:
    events = client.events(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        allow_gaps=True,
    )
    l2_snapshots = client.l2_snapshots(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        allow_gaps=True,
    )
    bbo = client.bbo(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        allow_gaps=True,
    )
    volume = client.volume(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        interval=bucket_interval,
        allow_gaps=True,
    )
    vwap = client.vwap(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        interval=bucket_interval,
        allow_gaps=True,
    )
    volatility = client.volatility(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        interval=bucket_interval,
        allow_gaps=True,
    )
    funding_rates = client.funding_rates(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        allow_gaps=True,
    )
    mark_prices = client.mark_prices(
        source=source,
        market=market,
        from_=window_start,
        to=window_end,
        allow_gaps=True,
    )

pd.DataFrame(
    {
        "method": [
            "events",
            "l2_snapshots",
            "bbo",
            "volume",
            "vwap",
            "volatility",
            "funding_rates",
            "mark_prices",
        ],
        "rows": [
            len(events),
            len(l2_snapshots),
            len(bbo),
            len(volume),
            len(vwap),
            len(volatility),
            len(funding_rates),
            len(mark_prices),
        ],
    }
)


## Normalize the responses into analysis frames

`events(...)` and `l2_snapshots(...)` both return standardized order-book payloads, while `bbo(...)`
and the bucketed methods already arrive in flatter shapes that map cleanly into DataFrames.


In [ ]:
def to_timestamp(series):
    return pd.to_datetime(series, unit="ms", utc=True)


def normalize_point_series(rows, value_column):
    frame = pd.json_normalize(rows)
    if frame.empty:
        return pd.DataFrame(columns=["timestamp", "series", value_column])
    frame["timestamp"] = to_timestamp(frame["timestamp"])
    rename_map = {}
    if "data.series" in frame.columns:
        rename_map["data.series"] = "series"
    if "data.value" in frame.columns:
        rename_map["data.value"] = value_column
    frame = frame.rename(columns=rename_map)
    if value_column not in frame.columns:
        data_value_columns = [
            column
            for column in frame.columns
            if column.startswith("data.") and column not in {"data.series"}
        ]
        if len(data_value_columns) == 1:
            frame = frame.rename(columns={data_value_columns[0]: value_column})
    if "series" not in frame.columns:
        frame["series"] = None
    return frame


events_df = pd.DataFrame(events)
events_df["timestamp"] = to_timestamp(events_df["timestamp"])
events_df["bid_levels"] = events_df["data"].apply(lambda payload: len(payload.get("bids", [])))
events_df["ask_levels"] = events_df["data"].apply(lambda payload: len(payload.get("asks", [])))

snapshots_df = pd.DataFrame(l2_snapshots)
snapshots_df["timestamp"] = to_timestamp(snapshots_df["timestamp"])
snapshots_df["bid_levels"] = snapshots_df["data"].apply(lambda payload: len(payload.get("bids", [])))
snapshots_df["ask_levels"] = snapshots_df["data"].apply(lambda payload: len(payload.get("asks", [])))

bbo_df = pd.DataFrame(bbo)
bbo_df["timestamp"] = to_timestamp(bbo_df["timestamp"])
bbo_df["mid_price"] = (bbo_df["bid_price"] + bbo_df["ask_price"]) / 2.0
bbo_df["spread"] = bbo_df["ask_price"] - bbo_df["bid_price"]
bbo_df["spread_bps"] = (bbo_df["spread"] / bbo_df["mid_price"]) * 10_000

volume_df = pd.DataFrame(volume).rename(columns={"volume": "bucket_volume"})
vwap_df = pd.DataFrame(vwap).rename(columns={"volume": "vwap_volume"})
volatility_df = pd.DataFrame(volatility)

bucket_df = (
    volume_df.merge(vwap_df, on="timestamp", how="outer")
    .merge(volatility_df, on="timestamp", how="outer")
    .sort_values("timestamp")
)
bucket_df["timestamp"] = to_timestamp(bucket_df["timestamp"])

funding_df = normalize_point_series(funding_rates, value_column="funding_rate")
mark_df = normalize_point_series(mark_prices, value_column="mark_price")

summary_df = pd.DataFrame(
    {
        "events": [len(events_df)],
        "snapshots": [len(snapshots_df)],
        "bbo_rows": [len(bbo_df)],
        "bucket_rows": [len(bucket_df)],
        "avg_spread_bps": [bbo_df["spread_bps"].mean()],
        "max_spread_bps": [bbo_df["spread_bps"].max()],
        "total_volume": [bucket_df["bucket_volume"].sum()],
        "funding_rows": [len(funding_df)],
        "mark_rows": [len(mark_df)],
    }
)

summary_df


In [ ]:
events_df[["timestamp", "type", "bid_levels", "ask_levels"]].head()


In [ ]:
bbo_df.head()


In [ ]:
bucket_df


## Point-series methods that are empty in the current public Lighter window

If `funding_rates(...)` or `mark_prices(...)` return rows in the future, the same code path below will
surface them. In the current public AAPL window they are empty, which is still useful to show because it
illustrates the method contract without making stale assumptions about coverage.


In [ ]:
{
    "funding_rates_empty": funding_df.empty,
    "mark_prices_empty": mark_df.empty,
}


## Visualize point-series methods when available

`mark_prices(...)` and `funding_rates(...)` are plotted separately because they are lower-frequency point
series rather than order-book snapshots. When the current public window does not expose rows, the chart
calls that out directly instead of failing silently.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

if mark_df.empty:
    axes[0].text(
        0.5,
        0.5,
        "No mark price rows in the selected public window",
        ha="center",
        va="center",
        transform=axes[0].transAxes,
    )
else:
    axes[0].plot(
        mark_df["timestamp"],
        mark_df["mark_price"],
        linewidth=1.4,
        color="#9467bd",
        label="mark price",
    )
    if not bbo_df.empty:
        axes[0].plot(
            bbo_df["timestamp"],
            bbo_df["mid_price"],
            linewidth=1.0,
            color="#1f77b4",
            alpha=0.7,
            label="mid price",
        )
    axes[0].legend(loc="upper left")
axes[0].set_ylabel("Price")
axes[0].set_title("Mark price")

if funding_df.empty:
    axes[1].text(
        0.5,
        0.5,
        "No funding rate rows in the selected public window",
        ha="center",
        va="center",
        transform=axes[1].transAxes,
    )
else:
    axes[1].plot(
        funding_df["timestamp"],
        funding_df["funding_rate"],
        linewidth=1.4,
        color="#8c564b",
    )
    axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.7)
axes[1].set_ylabel("Funding")
axes[1].set_xlabel("Time (UTC)")
axes[1].set_title("Funding rate")

plt.tight_layout()
plt.show()


## Visualize book quality and trade-derived buckets

The first panel shows the top-of-book mid price and per-minute VWAP, the second tracks spread in basis
points, and the last panel combines per-minute volume with realized volatility.


In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(14, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 2, 2]},
)

axes[0].plot(bbo_df["timestamp"], bbo_df["mid_price"], linewidth=1.2, color="#1f77b4", label="mid price")
axes[0].plot(bucket_df["timestamp"], bucket_df["vwap"], linewidth=1.2, color="#ff7f0e", label="1m vwap")
axes[0].set_ylabel("Price")
axes[0].set_title(f"{source}:{market} ({base_asset}) top-of-book and bucketed VWAP")
axes[0].legend(loc="upper left")

axes[1].plot(bbo_df["timestamp"], bbo_df["spread_bps"], linewidth=1.0, color="#2ca02c")
axes[1].set_ylabel("Spread (bps)")
axes[1].set_title("Best-bid-offer spread")

axes[2].bar(bucket_df["timestamp"], bucket_df["bucket_volume"], width=0.00045, color="#4c78a8", alpha=0.8, label="1m volume")
axes[2].set_ylabel("Volume")
axes[2].set_title("Per-minute volume and realized volatility")

volatility_axis = axes[2].twinx()
volatility_axis.plot(
    bucket_df["timestamp"],
    bucket_df["volatility"],
    linewidth=1.2,
    color="#d62728",
    label="1m realized volatility",
)
volatility_axis.set_ylabel("Volatility")

handles_1, labels_1 = axes[2].get_legend_handles_labels()
handles_2, labels_2 = volatility_axis.get_legend_handles_labels()
axes[2].legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left")
axes[2].set_xlabel("Time (UTC)")

plt.tight_layout()
plt.show()
